# Drop Detector Dataset Builder (Human-in-the-Loop)

This notebook is helpful to build the dataset for the drop detector. It runs a base version of the model on a specified music folder and then allows to interactively review best drop candidates and quickly create a labeled csv.

In [ ]:
# Imports & config

from pathlib import Path
from typing import Optional, Iterable, Tuple, List, Dict
import os
import re
import struct

import numpy as np
import pandas as pd
import joblib
import librosa

from mutagen import File as MutagenFile
from mutagen.id3 import ID3, TXXX
from mutagen.mp3 import MP3
from mutagen.mp4 import MP4
from mutagen.flac import FLAC
from mutagen.oggvorbis import OggVorbis
from mutagen.asf import ASF

from IPython.display import Audio, display, clear_output

# ---- Paths ----
MODEL_PATH = Path(
    "/Users/Administrateur/Software/Genre Classifier/drop detector/drop_detector_model.joblib"
)
MUSIC_ROOT = Path(
    "/Users/Administrateur/Software/Genre Classifier/drop detector/training_dataset"
)
OUTPUT_CSV = Path("./drop_annotations.csv")

# ---- Audio & model params (must match training) ----
SR = 22050  # resample rate for librosa.load
SEG_DEFAULT = 12.0  # will be overwritten by model params if present
WIN_DEFAULT = 4.0
BOOST_DEFAULT = 0.0

# Min score for candidates to be reviewed
MIN_SCORE = 0.5

# Length of snippet played around each candidate (seconds)
SNIPPET_SECONDS = 8.0

# Audio extensions
AUDIO_EXTS = {".mp3", ".wav", ".flac", ".m4a", ".aac", ".ogg", ".aiff", ".aif", ".wma"}

print("Config loaded.")


In [ ]:
# Helpers


def _first_nonempty(val) -> Optional[str]:
    """Normalize common tag value shapes (list/tuple/bytes/str) -> clean str or None."""
    if val is None:
        return None
    if isinstance(val, (list, tuple)):
        for v in val:
            s = _first_nonempty(v)
            if s:
                return s
        return None
    try:
        s = str(val).strip()
    except Exception:
        return None
    return s or None


def _get_any(dct, keys: Iterable[str]) -> Optional[str]:
    for k in keys:
        if k in dct:
            s = _first_nonempty(dct[k])
            if s:
                return s
    return None


def read_tags(path: Path) -> Tuple[Optional[str], Optional[str]]:
    """
    Return (genre, comment) using robust, format-specific rules.
    Same as in your original code, kept for completeness.
    """
    try:
        m = MutagenFile(path.as_posix())
        if m is None:
            return None, None

        if isinstance(m, MP3) or isinstance(m.tags, ID3):
            tags: ID3 = m.tags if isinstance(m.tags, ID3) else ID3(path.as_posix())
            genre = None
            tcon = tags.getall("TCON") if tags else []
            if tcon:
                for frame in tcon:
                    genre = _first_nonempty(getattr(frame, "text", None))
                    if genre:
                        break
            comment = None
            comms = tags.getall("COMM") if tags else []
            for frame in comms:
                txt = _first_nonempty(getattr(frame, "text", None))
                if txt:
                    comment = txt
                    break
            return genre, comment

        if isinstance(m, MP4):
            tags = m.tags or {}
            genre = _get_any(tags, ["\u00a9gen", "gnre"])
            comment = _get_any(
                tags, ["\u00a9cmt", "desc", "ldes", "----:com.apple.iTunes:COMMENT"]
            )
            return genre, comment

        if isinstance(m, FLAC):
            tags = m.tags or {}

            def _flac_get(keys):
                for k in keys:
                    if k in tags:
                        v = tags.get(k)
                        s = _first_nonempty(v)
                        if s:
                            return s
                low = {k.lower(): v for k, v in tags.items()}
                for k in keys:
                    if k.lower() in low:
                        s = _first_nonempty(low[k.lower()])
                        if s:
                            return s
                return None

            genre = _flac_get(["GENRE", "Genre"])
            comment = _flac_get(
                ["COMMENT", "COMMENTS", "DESCRIPTION", "Comment", "Description"]
            )
            return genre, comment

        if isinstance(m, OggVorbis):
            tags = m.tags or {}
            genre = _get_any(tags, ["genre", "GENRE"]) or _get_any(
                {k.lower(): v for k, v in tags.items()}, ["genre"]
            )
            comment = _get_any(
                tags, ["comment", "comments", "description"]
            ) or _get_any(
                {k.lower(): v for k, v in tags.items()},
                ["comment", "comments", "description"],
            )
            return genre, comment

        if isinstance(m, ASF):
            tags = m.tags or {}
            genre = _get_any(tags, ["WM/Genre"])
            comment = _get_any(tags, ["WM/Comments", "Description"])
            return genre, comment

        tags = getattr(m, "tags", {}) or {}
        genre = _get_any(tags, ["genre", "GENRE", "\u00a9gen"])
        comment = _get_any(
            tags, ["comment", "COMMENTS", "\u00a9cmt", "desc", "DESCRIPTION"]
        )
        return genre, comment

    except Exception:
        return None, None


# Filename parsing (artist - title)
dashes = re.compile(r"\s*[-–—]\s*")


def parse_filename(path: Path):
    stem = path.stem.strip()
    stem = re.sub(r"\([^)]*\)", "", stem)
    stem = re.sub(r"\[[^\]]*\]", "", stem)
    stem = re.sub(r"\s+", " ", stem).strip()

    parts = dashes.split(stem, maxsplit=1)
    if len(parts) == 2:
        artist = parts[0].strip() or None
        title = parts[1].strip()
    else:
        artist = None
        title = stem
    return artist, title


def read_drop_time(path: Path) -> Optional[float]:
    """Read existing DROP_TIME tag from mp3/wav, if present."""
    ext = path.suffix.lower()
    if ext == ".mp3":
        audio = MutagenFile(path.as_posix())
        if audio is None or not hasattr(audio, "tags") or audio.tags is None:
            return None
        for frame in audio.tags.getall("TXXX"):
            if frame.desc == "DROP_TIME":
                try:
                    return float(frame.text[0])
                except Exception:
                    return None
        return None
    elif ext == ".wav":
        try:
            with open(path, "rb") as f:
                content = f.read()
            i = content.find(b"drop")
            if i == -1:
                return None
            return struct.unpack("<f", content[i + 8 : i + 12])[0]
        except Exception:
            return None
    return None


print("Helpers ready.")

In [ ]:
# Segmentation and feature extraction


def _robust_slice(y, sr, center_s, win_s, offset=0.0):
    """Take a window around center_s + offset."""
    c = center_s + offset
    start = max(0, int((c - win_s / 2) * sr))
    end = min(len(y), int((c + win_s / 2) * sr))
    seg = y[start:end]
    min_len = int(sr * 0.5)
    if len(seg) == 0:
        return np.zeros(min_len, dtype=np.float32)
    if len(seg) < min_len:
        pad = min_len - len(seg)
        seg = np.pad(seg, (0, pad), mode="reflect" if len(seg) > 1 else "constant")
    return seg


def _stats_vec(x):
    """Summarize a 1D or 2D feature array into a fixed-length vector."""
    if x.ndim == 1:
        return np.array(
            [np.mean(x), np.std(x), np.median(x), np.max(x), np.min(x)],
            dtype=np.float32,
        )
    m = np.mean(x, axis=1)
    s = np.std(x, axis=1)
    p = np.percentile(x, [10, 25, 50, 75, 90], axis=1)
    return np.concatenate([m, s, p.flatten()]).astype(np.float32)


def _bass_boost_simple(y, sr, low=50, high=150, gain_db=0.0):
    """Gentle FFT-domain bass boost."""
    if gain_db == 0.0:
        return y
    Y = np.fft.rfft(y)
    freqs = np.fft.rfftfreq(len(y), 1 / sr)
    mask = (freqs >= low) & (freqs <= high)
    Y[mask] *= 10 ** (gain_db / 20.0)
    yb = np.fft.irfft(Y, n=len(y))
    return np.clip(yb, -1.0, 1.0)


def _segment_track(y, sr, seg_s, hop_length=512, boost_db=0.0):
    """Chroma+MFCC embedding -> agglomerative segmentation -> boundary times."""
    yb = _bass_boost_simple(y, sr, gain_db=boost_db)
    chroma = librosa.feature.chroma_cqt(y=yb, sr=sr, hop_length=hop_length)
    mfcc = librosa.feature.mfcc(y=yb, sr=sr)
    X = np.vstack(
        [librosa.util.normalize(chroma, axis=1), librosa.util.normalize(mfcc, axis=1)]
    )
    duration = len(yb) / sr
    n_segments = max(4, int(max(1, duration / float(seg_s))))
    bound_idxs = librosa.segment.agglomerative(X, k=n_segments, axis=1)
    times = librosa.frames_to_time(bound_idxs, sr=sr, hop_length=hop_length)
    return np.unique(np.concatenate([[0.0], times, [duration]])).astype(float)


def _features_context(y, sr, t, win_s, hop_length=512, boost_db=0.0):
    """Pre / Mid / Post contextual features + deltas."""
    yb = _bass_boost_simple(y, sr, gain_db=boost_db)

    pre = _robust_slice(yb, sr, t, win_s, offset=-win_s / 2)
    mid = _robust_slice(yb, sr, t, win_s, offset=0.0)
    post = _robust_slice(yb, sr, t, win_s, offset=+win_s / 2)

    def feat(seg):
        mel = librosa.feature.melspectrogram(y=seg, sr=sr, n_mels=64, power=2.0)
        logmel = librosa.power_to_db(mel, ref=np.max)
        mfcc = librosa.feature.mfcc(S=librosa.power_to_db(mel), sr=sr, n_mfcc=13)
        onset_env = librosa.onset.onset_strength(y=seg, sr=sr, hop_length=hop_length)
        tempogram = librosa.feature.tempogram(y=seg, sr=sr, hop_length=hop_length)
        rolloff = librosa.feature.spectral_rolloff(y=seg, sr=sr)
        contrast = librosa.feature.spectral_contrast(y=seg, sr=sr)
        zcr = librosa.feature.zero_crossing_rate(y=seg)
        rms = librosa.feature.rms(y=seg)

        S = np.abs(librosa.stft(seg, n_fft=2048, hop_length=hop_length)) ** 2
        freqs = librosa.fft_frequencies(sr=sr, n_fft=2048)
        mask = (freqs >= 50) & (freqs <= 150)
        low_pow = (
            np.sum(S[mask], axis=0)
            if np.any(mask)
            else np.zeros(S.shape[1], dtype=np.float32)
        )

        return np.concatenate(
            [
                _stats_vec(logmel),
                _stats_vec(mfcc),
                _stats_vec(onset_env),
                _stats_vec(tempogram),
                _stats_vec(rolloff),
                _stats_vec(contrast),
                _stats_vec(zcr),
                _stats_vec(rms),
                _stats_vec(low_pow),
            ]
        )

    fp, fm, fpost = feat(pre), feat(mid), feat(post)
    d_pm = fpost - fp
    d_cm = fm - fp
    d_pc = fpost - fm
    return np.concatenate([fp, fm, fpost, d_pm, d_cm, d_pc]).astype(np.float32)


print("Segmentation & feature functions ready.")

In [ ]:
# Load model bundle and parameters

bundle = joblib.load(MODEL_PATH)
clf = bundle["model"]
params = bundle.get("params", {})

seg_s = float(params.get("seg_s", SEG_DEFAULT))
win_s = float(params.get("win", WIN_DEFAULT))
boost_db = float(params.get("boost_db", BOOST_DEFAULT))

print("Model loaded.")
print("seg_s =", seg_s, "win_s =", win_s, "boost_db =", boost_db)

In [ ]:
# Candidate generation for a single track


def get_drop_candidates_for_track(
    audio_path: Path,
    clf,
    seg_s: float,
    win_s: float,
    boost_db: float,
    sr: int = SR,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Returns:
      boundaries: array of candidate times (seconds)
      scores: same shape, model score for each boundary
    """
    y, sr_loaded = librosa.load(audio_path.as_posix(), sr=sr, mono=True)
    boundaries = _segment_track(y, sr_loaded, seg_s=seg_s, boost_db=boost_db)
    feats = [
        _features_context(y, sr_loaded, float(t), win_s=win_s, boost_db=boost_db)
        for t in boundaries
    ]
    X = np.vstack(feats)

    if hasattr(clf, "decision_function"):
        scores = clf.decision_function(X)
    elif hasattr(clf, "predict_proba"):
        proba = clf.predict_proba(X)
        scores = proba[:, 1] if proba.ndim == 2 else proba
    else:
        preds = clf.predict(X)
        scores = (preds == 1).astype(float)

    return boundaries, scores

In [ ]:
# Scan music folder & load existing annotations


def scan_audio_paths(root: Path) -> List[Path]:
    paths = []
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in AUDIO_EXTS:
            paths.append(p)
    paths = sorted(paths)
    print(f"Found {len(paths)} audio files under {root}")
    return paths


all_paths = scan_audio_paths(MUSIC_ROOT)

if OUTPUT_CSV.exists():
    existing_df = pd.read_csv(OUTPUT_CSV)
    print(f"Existing annotation file found with {len(existing_df)} rows.")
else:
    existing_df = pd.DataFrame(columns=["path", "drop_time", "score", "source"])

existing_paths_with_labels = set(existing_df["path"].unique())
print(
    f"{len(existing_paths_with_labels)} tracks already have at least one labeled drop."
)

In [ ]:
# Interactive labeling of candidates

annotations: List[Dict] = existing_df.to_dict(orient="records")


def already_labeled(path: Path, t: float, tolerance: float = 0.25) -> bool:
    """Check if a (path, t) pair is already in annotations within `tolerance` seconds."""
    p = path.as_posix()
    for row in annotations:
        if row["path"] == p and abs(row["drop_time"] - t) <= tolerance:
            return True
    return False


def label_track(path: Path):
    """
    Interactive labeling for a single track:
      - reads existing DROP_TIME tag (if any)
      - generates candidates above MIN_SCORE
      - plays snippet for each candidate
      - asks user 'y'/'n' to accept or reject
      - records all confirmed drops into `annotations`
    """
    clear_output(wait=True)
    print(f"=== Track ===\n{path}\n")

    # Basic info
    artist, title = parse_filename(path)
    if artist or title:
        print(f"Parsed: artist={artist}, title={title}")

    # Existing metadata drop
    tag_drop = read_drop_time(path)
    if tag_drop is not None:
        print(f"Existing metadata drop tag: {tag_drop:.2f} s")
        if not already_labeled(path, tag_drop):
            annotations.append(
                {
                    "path": path.as_posix(),
                    "drop_time": float(tag_drop),
                    "score": np.nan,
                    "source": "tag",
                }
            )
    else:
        print("No DROP_TIME tag in metadata.")

    # Generate candidates
    print("\nComputing model candidates…")
    boundaries, scores = get_drop_candidates_for_track(
        path, clf, seg_s=seg_s, win_s=win_s, boost_db=boost_db, sr=SR
    )

    # Load full audio once for snippets
    y, sr_loaded = librosa.load(path.as_posix(), sr=SR, mono=True)

    # Filter by score threshold
    mask = scores >= MIN_SCORE
    cand_times = boundaries[mask]
    cand_scores = scores[mask]

    if len(cand_times) == 0:
        print(f"No candidate drops above score {MIN_SCORE}.")
        return

    # Sort candidates by score descending
    order = np.argsort(cand_scores)[::-1]
    cand_times = cand_times[order]
    cand_scores = cand_scores[order]

    print(f"Reviewing {len(cand_times)} candidates (score >= {MIN_SCORE}).\n")
    print("Controls:")
    print("  y = confirm drop")
    print("  n / Enter / anything else = reject")
    print("  s = skip remaining candidates for this track\n")

    for idx, (t, s) in enumerate(zip(cand_times, cand_scores), start=1):
        clear_output(wait=True)
        print(f"=== Track ===\n{path}\n")
        print(
            f"Candidate {idx}/{len(cand_times)}  |  t = {t:.2f} s  |  score = {s:.3f}"
        )
        if tag_drop is not None:
            print(f"(Known tag drop at {tag_drop:.2f} s)")

        # Generate snippet
        seg = _robust_slice(y, sr_loaded, center_s=float(t), win_s=SNIPPET_SECONDS)
        display(Audio(seg, rate=sr_loaded))

        resp = input("Is this a TRUE drop? [y/N/s(skip track)] ").strip().lower()

        if resp == "s":
            print("Skipping remaining candidates for this track.")
            break

        if resp == "y":
            if not already_labeled(path, float(t)):
                annotations.append(
                    {
                        "path": path.as_posix(),
                        "drop_time": float(t),
                        "score": float(s),
                        "source": "model_candidate",
                    }
                )
                print("✓ Saved.")
            else:
                print("Already labeled, skipping duplicate.")
        else:
            print("✗ Rejected.")

    print("\nDone with this track.")


# Main loop over tracks
for i, path in enumerate(all_paths, start=1):
    if path.as_posix() in existing_paths_with_labels:
        # If you DON'T want to skip, comment this block out.
        print(f"[{i}/{len(all_paths)}] Skipping (already has labels): {path.name}")
        continue

    print(f"\n[{i}/{len(all_paths)}] Now labeling: {path.name}")
    label_track(path)

    # Save progress after each track
    df_ann = pd.DataFrame(annotations)
    df_ann = df_ann.sort_values(["path", "drop_time"]).reset_index(drop=True)
    df_ann.to_csv(OUTPUT_CSV, index=False)
    print(f"Progress saved to {OUTPUT_CSV}\n")

print("All tracks processed.")